<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/BaselineTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --force-reinstall numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 128.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-hea

In [1]:
import os
#os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import numpy as np
import gc

from google.colab import drive
from google.colab import runtime

drive.mount("/content/drive", force_remount = False)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

import torchvision
from tqdm.notebook import tqdm

import random

from torch.utils.data import Dataset, DataLoader

#import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
#CUDA_version = torch.version.cuda.replace('.', '')

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

from scipy.ndimage import maximum_filter
from scipy.spatial._qhull import ConvexHull

import sys
import json

#!pip install import-ipynb
#import import_ipynb

#!pip install open3d plotly
#import open3d as o3d

import matplotlib.pyplot as plt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!npx degit google-research-datasets/Objectron/objectron objectron

!pip install --force-reinstall opencv-python-headless==4.9.0.80 &> /dev/null
!pip install nuscenes-devkit &> /dev/null

⠙⠹⠸⠼⠴⠦Need to install the following packages:
degit@3.10.1
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦> cloned google-research-datasets/Objectron#HEAD to objectron
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.3
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.3
npm notice To update run: npm install -g npm@12.2.3
npm notice
⠙

In [3]:
!mkdir -p /content/nuscenes

!tar -xzf /content/drive/MyDrive/LiDARFusion/v1.0-mini.tgz -C /content/nuscenes

In [4]:
sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box as BoxIoU

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, Box
from nuscenes.eval.detection.utils import category_to_detection_name
from nuscenes.utils.geometry_utils import points_in_box
from nuscenes.utils.splits import create_splits_scenes

nusc_root = "/content/nuscenes"

nusc = NuScenes(version='v1.0-mini', dataroot=nusc_root, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.377 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


In [5]:
from pyquaternion import Quaternion

sys.path.insert(0, '/content/drive/MyDrive/LiDARFusion')

sys.path.append('/content/objectron/')

from voxel_pointnet2 import PointNetSetAbstraction

from centerpoint import CenterPoint

from lidar_baseline import PillarsEncoder

import lidartrainlibrary as ltb

import nusceneseval as eval

#import CenterPointHead

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pickle

dict_file_path = '/content/drive/MyDrive/LiDARFusion/gt_dict.pkl'

with open(dict_file_path, 'rb') as f:
    gt_paste = pickle.load(f)


In [7]:
class_names = ["barrier", "bicycle", "bus", "car", "construction_vehicle", "motorcycle", "pedestrian", "traffic_cone", "trailer", "truck"]

def check_bev_iou(new_box, existing):
  new_box_corners = new_box.corners()[0:2, [0,1,4,5]]

  for box in existing:
    corners = box.corners()[0:2,[0,1,4,5]]

    corners_x , corners_y = corners

    for coords in new_box_corners.T:
      x,y = coords

      if corners_x[corners_x<x].size > 0 and corners_x[corners_x>x].size > 0:
        if corners_y[corners_y<y].size >0 and corners_y[corners_y>y].size >0:

          return True

  return False

def sample_valid_placement(gt_boxes):
  x_min, y_min, x_max, y_max, z_min, z_max = ltb.get_boxes_max_min(gt_boxes)

  x_cand = np.random.uniform(x_min, x_max)

  y_cand = np.random.uniform(y_min, y_max)

  z_cand = np.random.uniform(z_min, z_max)

  return np.array([x_cand, y_cand, z_cand])

def gt_sampling(cloud, gt_boxes, gt_names, gt_dict, sample_groups): #sample_groups is dict of how many of each class to sample

  for cls, num_sample in sample_groups.items():
    candidates = random.sample(gt_dict[cls], num_sample)

    for cand in candidates:
      new_center = sample_valid_placement(gt_boxes)
      new_box = cand["box"]
      new_box.translate(new_center)

      detection_name = ltb.category_to_detection_name(new_box.name)
      new_box.name = detection_name

      label = ltb.category_to_label(new_box.name)
      new_box.label = label

      if check_bev_iou(new_box, gt_boxes):
        continue

      new_points = cand['points'].copy()
      new_points[:, :3] += new_center

      points = np.concatenate([cloud, new_points], axis=0)
      gt_boxes.append(new_box)
      gt_names.append(cls)

  return points, gt_boxes, gt_names

In [8]:
test_scene = nusc.scene[0]

token_0 = test_scene["first_sample_token"]

cloud_sample = nusc.get("sample", token_0)

boxes = nusc.get_boxes(cloud_sample["data"]["LIDAR_TOP"])

points = ltb.load_sweeps(nusc, cloud_sample)

gt_boxes = ltb.process_boxes(nusc, cloud_sample, boxes, points)

gt_names = ltb.get_box_names(gt_boxes)

In [9]:
import time

class LidarDetectionDataset(Dataset): #contains only the nusc samples from which points/boxes are pulled

  def __init__(self, samples, gt_paste_samplegroups, pc_range = [-51.2, -51.2, 51.2, 51.2], val = False, voxel_size = 0.2):

    self.samples = samples

    self.sample_groups = gt_paste_samplegroups
    self.pc_range = pc_range
    self.val = val
    self.voxel_size = voxel_size

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    time_0 = time.time()

    sample_idx = self.samples[idx]
    token = sample_idx["token"]

    points = ltb.load_sweeps(nusc, sample_idx)
    boxes = nusc.get_boxes(sample_idx["data"]["LIDAR_TOP"])
    boxes = ltb.process_boxes(nusc, sample_idx, boxes, points)
    names = ltb.get_box_names(boxes)

    if self.val == False:
      points, boxes, names = gt_sampling(points, boxes, names, gt_paste, self.sample_groups)

    ltb.global_augmentation(points, boxes)

    points = ltb.cloud_filter_pcrange(points, pc_range = [self.pc_range[0], self.pc_range[2]])
    points = torch.from_numpy(points)
    boxes, names = ltb.boxes_filter_pcrange(boxes, pc_range = [self.pc_range[0], self.pc_range[2]])

    centers, sigmas, labels, offset, size, z, rotation, index, mask, cat = ltb.target_generation_vt(boxes, pc_range = self.pc_range)

    reg_targets = [offset, size, z, rotation]

    return {
        "points": points,
        "gt_boxes": boxes,
        "gt_labels": names,
        "heatmap_centers": centers,
        "heatmap_labels": labels,
        "sigmas": sigmas,
        "reg_target": reg_targets,
        "ind": index,
        "targets_mask": mask,
        "cat": cat,
        "token": token,
        "points_batch": None,
        "box_batch": None
    }

In [10]:
def collate_fn(batch):
    points = torch.cat([b['points'] for b in batch], dim=0)

    heatmap_centers = torch.cat([b['heatmap_centers'] for b in batch], dim=0)
    heatmap_labels = torch.cat([b['heatmap_labels'] for b in batch], dim=0)
    sigmas = torch.cat([b['sigmas'] for b in batch], dim=0)

    reg_target = [b['reg_target'] for b in batch]
    reg_mask = torch.stack([b['targets_mask'] for b in batch], dim=0)
    ind = torch.stack([b['ind'] for b in batch], dim=0)
    cat = torch.stack([b['cat'] for b in batch], dim=0)

    # gt_boxes/gt_labels stay as a list of variable-length tensors —
    # don't pad these for loss, only pull them out for stage-2 / eval
    gt_boxes = [b['gt_boxes'] for b in batch]
    gt_labels = [b['gt_labels'] for b in batch]
    tokens = [b['token'] for b in batch]

    points_batch_idx = []
    box_batch_idx = []
    for i, b in enumerate(batch):
        batch_p = torch.full((b["points"].shape[0],), i)
        batch_b = torch.full((len(b["gt_boxes"]),), i)
        points_batch_idx.append(batch_p)
        box_batch_idx.append(batch_b)
    points_batch = torch.cat(points_batch_idx)
    box_batch = torch.cat(box_batch_idx)

    return {
        'points': points, 'heatmap_centers': heatmap_centers, "heatmap_labels": heatmap_labels,"sigmas": sigmas,
        "reg_target": reg_target, 'targets_mask': reg_mask, 'points_batch': points_batch, 'box_batch': box_batch,
        'ind': ind, 'cat': cat, 'gt_boxes': gt_boxes, 'gt_labels': gt_labels, 'tokens': tokens
    }

In [11]:
splits_scenes = create_splits_scenes()
train_names = splits_scenes['mini_train']

samples = ltb.get_samples(train_names, nusc)
sample_groups = {"car": 2, "pedestrian": 1, "construction_vehicle": 2, "motorcycle": 2, "trailer": 2}

lidarset = LidarDetectionDataset(samples, sample_groups)

loader = DataLoader(lidarset, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)

In [25]:
import time

class LIDARBaseline(nn.Module):

  def __init__(self, encoder, extracter, in_dims = [512,512], pc_range = [-51.2, 51.2]):
    super().__init__()

    self.encoder = encoder
    self.extracter = extracter
    self.in_dims = in_dims
    self.C = encoder.out_dim
    self.pc_range = pc_range
    self.pillar_size = (self.pc_range[1]-self.pc_range[0])/ self.in_dims[0]

  def get_batch_ids(self, coords, B):

    id_list = []
    for i in range(B):
      B_ids = torch.where(coords[:, 0:1] == i)[0]
      id_list.append(B_ids)

    return id_list

  def forward(self, pillars, coords, mask, B=4, train = False, val = False, sample_tokens = None):

    boxes_list = []
    hm_list = []
    s1_centers_list = []
    refine_list = []
    I_list = []
    maps_list = []

    if val == True:
      tokens_list = []

    out = self.encoder(pillars, mask)
    canvas = torch.zeros((B, self.C, self.in_dims[0], self.in_dims[1]),device = pillars.device, dtype = out.dtype)
    b, y, x = coords[:, 0].long(), coords[:, 2].long(), coords[:, 1].long()

    canvas[b, :, y, x] = out

    for j in range(B):
      feature_map = canvas[j]

      boxes, centers, dvec, I_0, regression_maps, heat_logits = self.extracter(feature_map)

      boxes_list.append(boxes)
      hm_list.append(heat_logits)
      s1_centers_list.append(centers)
      refine_list.append(dvec)
      I_list.append(I_0)
      maps_list.append(regression_maps)

      if val == True:
        tokens_list.append(sample_tokens[j])

    heatmap_out = torch.stack(hm_list,dim=0)

    if train == True:

      return {"boxes": boxes_list, "heatmap": heatmap_out, "stage1_centers": s1_centers_list, "refine_input": refine_list, "I_input": I_list, "regression_maps": maps_list}

    if val == True:

      return boxes_list, tokens_list

    return boxes

Encoder = PillarsEncoder(extra_features=2)
BoxHead = CenterPoint(512,512,256,10, K = 50) #[l, w, h, sin, cos, x,y,z ,k, I]
model = LIDARBaseline(Encoder, BoxHead)

In [26]:
def loss_hm(pred, target, alpha=2, beta=4):
    # pred, target: [B, C, H, W] — same shape, C = num classes

    pos_mask = (target == 1).float()
    neg_mask = 1 - pos_mask

    pos_mask = pos_mask.to(pred.device)
    neg_mask = neg_mask.to(pred.device)

    weights = torch.sigmoid(pred).clamp(-1e-4, 1e-4)

    pos_loss = F.logsigmoid(pred) * (1 - weights).pow(alpha) * pos_mask
    neg_loss = F.logsigmoid(-pred) * pred.pow(alpha) * (1 - target).pow(beta) * neg_mask

    return -(pos_loss.sum() + neg_loss.sum()) / pos_mask.sum().clamp(min=1)

class LidarLoss(nn.Module):

  def __init__(self, alpha = 2, beta = 4, lambda_reg = 0.25, hm_dims = 512):
    super(LidarLoss, self).__init__()
    self.alpha = alpha
    self.beta = beta
    self.lambda_reg = lambda_reg
    self.l1loss = nn.L1Loss()
    self.offsetl1 = nn.L1Loss(reduction="none")
    self.conf_loss = nn.BCEWithLogitsLoss()
    self.dims = hm_dims

  @torch.no_grad()
  def match_candidates_to_gt(self, stage1_centers, gt_boxes, dist_thresh=2.0):
    matched_gt_idx = torch.full(((stage1_centers.shape[0]),), -1, dtype=torch.long, device = stage1_centers.device)

    if gt_boxes.shape[0] == 0 or stage1_centers.shape[0] == 0:
      return matched_gt_idx

    for i in range(stage1_centers.shape[0]):
      cand = stage1_centers[i]
      dists = torch.linalg.norm(gt_boxes[:, 5:7] - cand[5:7], dim=1)  # BEV center distance
      min_dist, min_idx = dists.min(dim=0)
      if min_dist < dist_thresh:
        matched_gt_idx[i] = min_idx
    return matched_gt_idx  # -1 = unmatched, no refinement target

  @torch.no_grad()
  def generate_stage2_targets(self, stage1_centers, gt_boxes):
    matched_gt = self.match_candidates_to_gt(stage1_centers, gt_boxes)
    valid = matched_gt >= 0
    N = stage1_centers.shape[0]
    delta_targets = torch.zeros((N, 7), dtype = stage1_centers.dtype, device = stage1_centers.device)
    I_targets = torch.zeros(N, dtype = stage1_centers.dtype, device = stage1_centers.device)

    with torch.no_grad():

      c, g = stage1_centers[valid], gt_boxes[matched_gt[valid]]

      delta_center = g[:, 5:8] - c[:, 5:8]  # residual center offset
      delta_dim = g[:, :3] - c[:, :3]
      delta_rot = torch.atan2(g[:, 3:4], g[:, 4:5]) - torch.atan2(c[:, 3:4], c[:, 4:5])
      delta_rot = torch.atan2(torch.sin(delta_rot), torch.cos(delta_rot))  # wrap

      delta_dim, delta_rot, delta_center = delta_dim.to(delta_targets.dtype), delta_rot.to(delta_targets.dtype), delta_center.to(delta_targets.dtype)

      delta_targets[valid] = torch.cat([delta_dim, delta_rot, delta_center], dim = 1)

      iou_mat = ltb.IoU2(c, g)

      iou_0 = iou_mat.diagonal().to(I_targets.dtype)

      I_targets[valid] = (2* iou_0 - 0.5).clamp(0, 1)

    return delta_targets, I_targets

  def forward(self, heatmap_input, regression_maps, ind, mask, stage1_centers, refine_input, I_input, target_map, target_reg, gt_boxes): #batched inputs for all, heatmap_input and I_input are raw_logits

    stage1_loss = 0
    stage2_loss = 0
    empty_samples = 0

    for j in range(heatmap_input.shape[0]):

      if j >= mask.shape[0]:
        empty_samples += 1
        continue

      num_sample = mask[j].nonzero()[0]

      x = (ind[j][:num_sample] // self.dims).to(torch.int32)
      y = (ind[j][:num_sample] % self.dims).to(torch.int32)

      offsets = target_reg[j][0][x, y]
      sizes = target_reg[j][1][x, y]
      z = target_reg[j][2][x, y]
      rotations = target_reg[j][3][x, y]

      o_hat = regression_maps[j][0][x, y] #[2, num_sample]
      size_hat = regression_maps[j][1][x, y]
      z_hat = regression_maps[j][2][x, y]
      rot_hat = regression_maps[j][3][x, y]

      #[8, num_sample]
      reg_inputs = torch.cat([o_hat, size_hat, z_hat.unsqueeze(1), rot_hat], dim = 1)
      target_regressed = torch.cat([offsets, sizes, z.unsqueeze(1), rotations], dim = 1).to(reg_inputs.device)

      box_tensor = ltb.boxes_to_tensor(gt_boxes[j]).to(stage1_centers[j].device)

      refine_mask = self.match_candidates_to_gt(stage1_centers[j], box_tensor)
      refine_mask = (refine_mask >= 0).float()
      refine_mask = refine_mask.to(heatmap_input.device)
      refine_target, I_targets = self.generate_stage2_targets(stage1_centers[j], box_tensor)

      target_map = target_map.to(heatmap_input.device)
      refine_in = refine_input[j].to(heatmap_input.device)
      I_in = I_input[j].to(heatmap_input.device)

      heatmap_loss = loss_hm(heatmap_input[j], target_map[j], alpha = self.alpha, beta = self.beta)

      reg_loss = self.l1loss(reg_inputs, target_regressed)

      reg_loss = reg_loss * self.lambda_reg

      stage1_loss += heatmap_loss + reg_loss

      offset_loss = self.offsetl1(refine_in, refine_target)
      offset_loss = (offset_loss * refine_mask.unsqueeze(-1)).sum() / (refine_mask.sum() + 1e-4)

      confidence_loss = self.conf_loss(I_in, I_targets)

      stage2_loss += offset_loss + confidence_loss

    stage1_loss = stage1_loss / (heatmap_input.shape[0] - empty_samples)
    stage2_loss = stage2_loss / (heatmap_input.shape[0] - empty_samples)

    return stage1_loss + stage2_loss

loss_lidar = LidarLoss()

In [27]:
device = "cuda"
model = model.to(device)
model.train()

num_model = 0

#state_dict = torch.load(f"/content/drive/MyDrive/LiDARFusion/lidarbaseline/lrwarmup_model_{num_model}.pt", weights_only=True)
#model.load_state_dict(state_dict)

In [28]:
lr = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr)
warmup_steps = 15

scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1 + (0.1*num_model),
    end_factor=1.0,
    total_iters=warmup_steps - num_model
)

In [ ]:
batch = next(iter(loader))

In [ ]:
pillars, coords, mask = ltb.voxelize_cloud(batch["points"].to("cuda"), batch_idx = batch["points_batch"].to("cuda"))

In [ ]:
import time

scaler = torch.amp.GradScaler("cuda")
epochs = 31
start = num_model
device = "cuda"

for epoch in tqdm(range(epochs)):
    time0 = time.time()
    for batch in loader:

      optimizer.zero_grad()

      points = batch["points"]
      points = points.to(device)
      points_batch = batch["points_batch"].to(device)
      pillars, coords, mask = ltb.voxelize_cloud(points, batch_idx = points_batch)

      pillars, mask = pillars.to(device).float(), mask.to(device).float()

      h_cent, sigmas, h_labels = batch["heatmap_centers"].to(device), batch["sigmas"].to(device), batch["heatmap_labels"].to(device)
      heatmap = ltb.splat_gaussians(h_cent, sigmas, h_labels, batch_idx = batch["box_batch"].to(device), B = 4)

      with torch.amp.autocast("cuda", dtype=torch.bfloat16):

        out = model(pillars, coords, mask, B=4, train = True)

        loss = loss_lidar(out["heatmap"], out["regression_maps"], batch["ind"], batch["targets_mask"], out["stage1_centers"], out["refine_input"], out["I_input"], heatmap, batch["reg_target"], batch["gt_boxes"])

      scaler.scale(loss).backward()
      scaler.unscale_(optimizer)
      grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=35.0)

      scaler.step(optimizer)
      scaler.update()

    file_path = f"/content/drive/MyDrive/LiDARFusion/lidarbaseline/lrwarmup_model_{epoch+1+start}.pt"
    print(loss.item())
    if epoch % 10 == 0 and epoch > 0:
      torch.save(model.state_dict(), file_path)
    scheduler.step()

runtime.unassign()

  0%|          | 0/31 [00:00<?, ?it/s]

14938.478515625
17233.638671875
16767.5625
10573.376953125
23992.0625
11098.2236328125
10505.013671875
16334.234375
14570.92578125
11633.1181640625
6447.70703125
10227.7822265625
16293.5283203125
15520.7900390625
13763.62890625
10446.5966796875
9984.15625
23112.80078125
8372.7587890625
11594.4052734375
9163.0517578125
11723.8857421875
8377.8779296875
11742.029296875
11704.7685546875
15319.826171875
5680.5322265625
9642.626953125
9433.76171875
9426.470703125
6123.50537109375
10802.5595703125
10211.8623046875
8932.009765625
12573.728515625
15503.3583984375
8297.5859375
12311.2490234375
7355.27001953125
5720.615234375
7912.4443359375
5895.9990234375
8524.8330078125
4954.7109375
5735.76611328125
10203.87109375
5819.91845703125
2737.42724609375
4533.8935546875
2626.043212890625
4591.64013671875
3633.5966796875
3266.510498046875
2136.12158203125
2259.0009765625
1202.75927734375
3002.205810546875
2319.952880859375
2270.696533203125
3104.985107421875
1795.187744140625
1105.33837890625
1552.868

In [ ]:
splits_scenes = create_splits_scenes()
val_names = splits_scenes['mini_val']

val_samples = ltb.get_samples(val_names, nusc)

val_set = LidarDetectionDataset(val_samples, {}, val = True)
loader_val = DataLoader(val_set, batch_size=4, shuffle=True,
                     collate_fn=collate_fn, num_workers=2, pin_memory=True)

device = "cuda"
boxes_per_sample = []
all_tokens = []
num_model = 13

boxes_path = f"/content/drive/MyDrive/LiDARFusion/results/boxes{num_model}.pkl"
tokens_path = f"/content/drive/MyDrive/LiDARFusion/results/token{num_model}.json"

for batch in loader_val:
  with torch.no_grad():
    pillars, coords, mask, tokens = batch["pillars"], batch["coords"], batch["pillars_mask"], batch["tokens"]
    pillars, mask = pillars.to(device), mask.to(device)

    boxes, tokens = model(pillars, coords, mask, train = False, val = True, sample_tokens = tokens)

    boxes_per_sample.extend(boxes)
    all_tokens.extend(tokens)

cpu_boxes = []
for i in range(len(boxes_per_sample)):
  sample_0 = boxes_per_sample[i]
  cpu_sample_boxes = []

  for j in range(len(sample_0)):
    cpu_sample_boxes.append(sample_0[j].to("cpu"))

  cpu_boxes.append(cpu_sample_boxes)

with open(boxes_path, 'wb') as file:
  pickle.dump(cpu_boxes, file)
with open(tokens_path, 'w') as f:
  json.dump(all_tokens, f)


In [ ]:
num_model = 13

boxes_path = f"/content/drive/MyDrive/LiDARFusion/results/boxes{num_model}.pkl"
tokens_path = f"/content/drive/MyDrive/LiDARFusion/results/token{num_model}.json"

with open(boxes_path, 'rb') as f:
    cpu_boxes = pickle.load(f)
with open(tokens_path, 'r', encoding='utf-8') as file:
    all_tokens = json.load(file)

In [ ]:
out_path = f"/content/drive/MyDrive/LiDARFusion/results/results{num_model}.json"
output_dir = "/content/drive/MyDrive/LiDARFusion/results"

eval.results_to_json(nusc, cpu_boxes, all_tokens, out_path, use_camera = False)

In [ ]:
eval_results = eval.run_eval(nusc, out_path, output_dir)

In [ ]:
splits_scenes = create_splits_scenes()
val_names = splits_scenes['mini_val']

val_samples = ltb.get_samples(val_names, nusc)

with open(out_path, 'r', encoding='utf-8') as file:
    results = json.load(file)

sample_token = val_samples[0]['token']
ann_tokens = nusc.get('sample', sample_token)['anns']
gt_translations = [nusc.get('sample_annotation', t)['translation'] for t in ann_tokens]

pred = results["results"][sample_token][0]  # first predicted box for this sample
pred_t = np.array(pred['translation'])
dists = [np.linalg.norm(pred_t - np.array(g)) for g in gt_translations]
print("min distance to any GT box:", min(dists) if dists else None)

In [ ]:
results["results"][sample_token][0]

In [ ]:
out["refine_input"][0].device

In [ ]:
batch["reg_target"][1][2][1][511:512]

In [ ]:
test_data = lidarset.__getitem__(4)